# 10 - Comparison against random network models

A single number like "average clustering = 0.02" tells you nothing on its own. To decide whether Israel's public-transport station network is *unusual* in its clustering, *unusual* in its heterogeneity, or *unusual* in how far it sits from a small-world structure, you need null models: synthetic graphs of the **same size**, produced by a known mechanism. This notebook rebuilds (or loads) the real undirected station graph and compares it against four classic models - Erdos-Renyi `G(n, m)`, a configuration model on the real degree sequence, a Barabasi-Albert preferential-attachment graph, and a Watts-Strogatz small-world graph - in terms of degree distribution, clustering, and average shortest-path length.

**Research question:** which generative mechanism, if any, reproduces the structure of the real bus-and-rail station network - and what does the mismatch tell us about the forces that actually shape the network?

**Input**
- `outputs/nb/02_graph_construction/**` - the station graph saved by the `02_graph_construction` notebook (as GraphML or as an edge-list CSV), if it exists.
- `israel-public-transportation/stop_times.txt` - a fallback path only. If step 02 wasn't run, the notebook rebuilds the graph directly from the raw GTFS feed (816 MB, downloaded as needed).

**Output** (all files under `outputs/nb/10_network_model_comparison/`)
- `tables/network_model_comparison.csv` - one row per graph, with the node and edge counts actually achieved and all the structural metrics.
- `tables/degree_distributions.csv` - the full degree distribution (P(k) and CCDF) for each graph.
- `tables/degree_distribution_summary.csv` - degree moments, a tail-exponent estimate, and the KS distance from the real degree distribution.
- `tables/small_world_ratios.csv` - C/C_random, L/L_random, and the small-world index sigma.
- `figures/degree_distribution_comparison.png`, `figures/model_metric_comparison.png`, `figures/size_match_check.png`.

**A bug this notebook fixes.** The original script picked the Barabasi-Albert parameter as `m = round(edges / nodes) = 2`, which produces about 61k edges against the real 51.8k, and forced the Watts-Strogatz `k` to be even, which collapsed it to `k = 2` - a bare ring with about 30.5k edges. Comparing clustering and path length between graphs whose edge counts differ by 20-40% is meaningless: both quantities depend directly on edge density. Below, the two generators are replaced by size-matched versions that hit the real edge count exactly, and every model reports the node and edge counts it actually achieved, so any mismatch that remains (the configuration model necessarily loses a few edges) stays visible.

## 1. Setting up the environment

This cell lets the notebook run both on a local copy of the repository and in Google Colab. It installs only the packages that are actually missing, locates the repository root by searching for the GTFS folder named `israel-public-transportation` (and clones the repository if we're running in Colab), and creates the shared `outputs/nb` folder that every notebook in this series writes to.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Imports, tuning constants, and step folders

All the expensive parameters live here, so you can tune runtime without touching the analysis code:

- `PATH_SOURCE_SAMPLES = 128` - exact all-pairs shortest paths over about 30k nodes isn't feasible (about 4.6e8 pairs). We run BFS from 128 random sources per graph and average the distances we get. The cost is about 128 BFS sweeps x 5 graphs, a few seconds each; raising it to 512 improves accuracy at roughly 4x the cost.
- `CLUSTERING_TRIALS = 2000` - used only as a fallback if exact clustering is deemed too expensive (see `CLUSTERING_EXACT_BUDGET`).
- `CLUSTERING_EXACT_BUDGET` - an approximation of the compute cost (`sum of degree^2`). Below this value we compute clustering exactly, which is cheap at the current sparsity and removes sampling noise from the central comparison.
- `WS_REWIRE_P = 0.05` - the Watts-Strogatz rewiring probability used in the original script; kept for continuity.
- `SEED = 42` - every generator and every sampler is seeded with a fixed value, so the whole notebook is reproducible.

Following the project's output convention, this notebook writes only to its own step folder and doesn't touch `outputs/tables`, `outputs/figures`, or `outputs/rail`.

In [ ]:
_ensure('networkx', 'pandas', 'matplotlib', 'numpy')

import csv
import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

SEED = 42
PATH_SOURCE_SAMPLES = 128        # BFS sources per graph for path-length statistics
CLUSTERING_TRIALS = 2000         # fallback sample size for approximate clustering
CLUSTERING_EXACT_BUDGET = 5_000_000  # sum(degree^2) below which exact clustering is affordable
WS_REWIRE_P = 0.05               # Watts-Strogatz rewiring probability
K_MIN_TAIL = 4                   # lower cut-off for the descriptive tail-exponent estimate

STAGE = OUT / '10_network_model_comparison'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

PREV_STAGE = OUT / '02_graph_construction'

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
print('Stage output folder:', STAGE)

## 3. Where does the real graph come from?

This notebook is a consumer of the station graph built in the `02_graph_construction` notebook. To stay robust, we first check that step's folder for any graph artifact (a `.graphml`/`.gexf` file, or a CSV whose name contains the string `edge`). If step 02 wasn't run, we fall back to rebuilding the graph from the raw GTFS feed using exactly the same construction rule, so a reviewer can run this notebook standalone. The cell below only *decides* which path to take and prints the decision.

In [ ]:
def find_stage02_graph():
    '''Return a graph artifact saved by notebook 02, or None if stage 02 is absent.'''
    if not PREV_STAGE.exists():
        return None
    for pattern in ('**/*.graphml', '**/*.gexf'):
        for path in sorted(PREV_STAGE.glob(pattern)):
            return path
    for path in sorted(PREV_STAGE.glob('**/*.csv')):
        if 'edge' in path.name.lower():
            return path
    return None


GRAPH_SOURCE = find_stage02_graph()
if GRAPH_SOURCE is None:
    print('No artifact found in', PREV_STAGE)
    print('Falling back to rebuilding the stop graph from the raw GTFS feed.')
    print('(Run notebook 02_graph_construction first if you prefer to reuse its output.)')
else:
    print('Reusing stage 02 artifact:', GRAPH_SOURCE)

## 4. The raw feed (fallback path only)

The file `stop_times.txt` weighs 816 MB and is deliberately kept out of git. It's needed only when we have to rebuild the graph ourselves, so the download is skipped entirely when a step 02 artifact is found. The download takes a few minutes on the first run and is cached on disk afterward.

In [ ]:
STOP_TIMES = DATA / 'stop_times.txt'

if GRAPH_SOURCE is None:
    # stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
    _ensure('gdown')
    import gdown
    if not STOP_TIMES.exists():
        gdown.download(id='1V_yPAWXV6mGTFGrfiosah5LngcLZnviW',
                       output=str(STOP_TIMES), quiet=False)
    print('stop_times.txt:', round(STOP_TIMES.stat().st_size / 1024**2, 1), 'MB')
else:
    print('Raw feed not needed - the graph comes from stage 02.')

## 5. Building (or loading) the real undirected station graph

The modeling rule, copied word for word from the project's analysis script: **nodes are GTFS stops, and an edge connects two stops that follow one another within the same trip**. The feed is sorted by `trip_id` and then by `stop_sequence`, so a single streaming pass over the 15.7M rows is enough - we never hold the whole file in memory. Parallel segments are collapsed together and their counts summed into the edge attribute `frequency` (how many scheduled trip segments use the same stop pair). Directionality is dropped here because clustering, path length, and degree distribution are all defined on the undirected graph.

Expect about 30k nodes and about 52k edges. The streaming rebuild takes about 2-4 minutes; loading a step 02 artifact is almost instant.

In [ ]:
def gtfs_rows(path):
    '''Stream a GTFS csv file row by row (never loads the whole file).'''
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        yield from csv.DictReader(handle)


def build_undirected_stop_graph(stop_times_path):
    '''Consecutive stops within a trip become an undirected weighted edge.'''
    edge_counts = Counter()
    stop_use_counts = Counter()
    previous_trip_id = None
    previous_stop_id = None
    rows_seen = 0

    for row in gtfs_rows(stop_times_path):
        rows_seen += 1
        trip_id = row['trip_id']
        stop_id = row['stop_id']
        stop_use_counts[stop_id] += 1
        if (previous_trip_id == trip_id
                and previous_stop_id is not None
                and previous_stop_id != stop_id):
            edge_counts[(previous_stop_id, stop_id)] += 1
        previous_trip_id = trip_id
        previous_stop_id = stop_id

    graph = nx.Graph()
    graph.add_nodes_from(stop_use_counts)
    for (source, target), frequency in edge_counts.items():
        if graph.has_edge(source, target):
            graph[source][target]['frequency'] += frequency
        else:
            graph.add_edge(source, target, frequency=frequency)
    print(f'stop_times rows streamed: {rows_seen:,}')
    return graph


def load_edge_list_csv(path):
    '''Load an undirected graph from a stage-02 edge-list csv with flexible column names.'''
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    columns = {name.lower(): name for name in frame.columns}
    source_col = next((columns[c] for c in
                       ('source', 'source_stop_id', 'from_stop_id', 'from_stop', 'stop_a', 'u')
                       if c in columns), None)
    target_col = next((columns[c] for c in
                       ('target', 'target_stop_id', 'to_stop_id', 'to_stop', 'stop_b', 'v')
                       if c in columns), None)
    if source_col is None or target_col is None:
        raise ValueError(
            f'{path} does not look like an edge list (columns: {list(frame.columns)}). '
            'Re-run notebook 02_graph_construction, or delete the file so this notebook '
            'rebuilds the graph from stop_times.txt.')
    weight_col = next((columns[c] for c in ('trip_frequency', 'frequency', 'weight', 'trips', 'count')
                       if c in columns), None)
    graph = nx.Graph()
    for _, row in frame.iterrows():
        source, target = row[source_col], row[target_col]
        weight = int(float(row[weight_col])) if weight_col else 1
        if graph.has_edge(source, target):
            graph[source][target]['frequency'] += weight
        else:
            graph.add_edge(source, target, frequency=weight)
    return graph


if GRAPH_SOURCE is None:
    G = build_undirected_stop_graph(STOP_TIMES)
elif GRAPH_SOURCE.suffix == '.graphml':
    G = nx.Graph(nx.read_graphml(GRAPH_SOURCE))
elif GRAPH_SOURCE.suffix == '.gexf':
    G = nx.Graph(nx.read_gexf(GRAPH_SOURCE))
else:
    G = load_edge_list_csv(GRAPH_SOURCE)

G.remove_edges_from(nx.selfloop_edges(G))
N_REAL = G.number_of_nodes()
M_REAL = G.number_of_edges()
if N_REAL == 0 or M_REAL == 0:
    raise RuntimeError('The loaded stop graph is empty - check the stage 02 artifact or the GTFS feed.')
print(f'Real transit graph: {N_REAL:,} nodes, {M_REAL:,} undirected edges, '
      f'average degree {2 * M_REAL / N_REAL:.3f}')

## 6. Measurement helpers (identical treatment for every graph)

For the comparison to be fair, every graph - real or synthetic - has to be measured by exactly the same code, the same sample size, and the same seed. The helpers below do that:

- `sampled_path_statistics` runs BFS from a fixed number of random sources **within the largest connected component** (an unreachable pair has infinite distance, which would make the average undefined) and averages the distances it gets. The largest observed distance is reported as a diameter estimate - it's a lower bound on the true diameter.
- `average_clustering_value` prefers the exact average local clustering coefficient, and falls back to networkx's sampling estimator only when the graph is dense enough to slow the exact computation down. The method used is recorded in the results table so nothing is hidden.
- `graph_property_row` produces one row of the comparison table, and deliberately starts with `nodes`, `edges`, and the ratios against the real graph, so any size mismatch is the first thing the reader sees.

In [ ]:
def largest_component_subgraph(graph):
    '''Copy of the largest connected component.'''
    largest_nodes = max(nx.connected_components(graph), key=len)
    return graph.subgraph(largest_nodes).copy()


def sampled_path_statistics(graph, source_samples, seed):
    '''Approximate average shortest path and diameter from sampled BFS sources.'''
    if graph.number_of_nodes() == 0 or source_samples <= 0:
        return {'approx_average_shortest_path': None, 'approx_diameter': None}
    component = largest_component_subgraph(graph)
    rng = random.Random(seed)
    nodes = list(component.nodes)
    sources = rng.sample(nodes, min(source_samples, len(nodes)))

    distance_sum = 0
    reached_pairs = 0
    max_distance = 0
    for source in sources:
        lengths = nx.single_source_shortest_path_length(component, source)
        for target, distance in lengths.items():
            if target == source:
                continue
            distance_sum += distance
            reached_pairs += 1
            max_distance = max(max_distance, distance)
    if reached_pairs == 0:
        return {'approx_average_shortest_path': None, 'approx_diameter': None}
    return {'approx_average_shortest_path': distance_sum / reached_pairs,
            'approx_diameter': max_distance}


def average_clustering_value(graph, trials, seed):
    '''Exact average clustering when affordable, sampled estimate otherwise.'''
    cost_proxy = sum(degree * degree for _, degree in graph.degree())
    if cost_proxy <= CLUSTERING_EXACT_BUDGET:
        return nx.average_clustering(graph), 'exact'
    sampled = nx.algorithms.approximation.average_clustering(
        graph, trials=min(trials, graph.number_of_nodes()), seed=seed)
    return sampled, f'sampled_{trials}'


def graph_property_row(name, graph, seed=SEED):
    '''One row of the model-comparison table, sizes first so mismatches are visible.'''
    n = graph.number_of_nodes()
    m = graph.number_of_edges()
    degrees = [degree for _, degree in graph.degree()]
    component_sizes = sorted((len(c) for c in nx.connected_components(graph)), reverse=True)
    path_stats = sampled_path_statistics(graph, PATH_SOURCE_SAMPLES, seed)
    clustering, clustering_method = average_clustering_value(graph, CLUSTERING_TRIALS, seed)
    return {
        'model': name,
        'nodes': n,
        'edges': m,
        'node_ratio_vs_real': n / N_REAL,
        'edge_ratio_vs_real': m / M_REAL,
        'average_degree': 2 * m / n if n else 0.0,
        'max_degree': max(degrees) if degrees else 0,
        'degree_std': float(np.std(degrees)) if degrees else 0.0,
        'density': nx.density(graph) if n > 1 else 0.0,
        'connected_components': len(component_sizes),
        'largest_component_share': component_sizes[0] / n if n else 0.0,
        'average_clustering': clustering,
        'clustering_method': clustering_method,
        'transitivity': nx.transitivity(graph),
        'approx_average_shortest_path': path_stats['approx_average_shortest_path'],
        'approx_diameter': path_stats['approx_diameter'],
    }


print('Measurement helpers ready.')

## 7. The null models - and the size-match fix

Four models, each answering a different "what if" question:

| Model | Mechanism | What it controls for |
|---|---|---|
| Erdos-Renyi `G(n, m)` | edges placed uniformly at random | size and density only |
| configuration model | random rewiring of the **real degree sequence** | size, density, *and* degree distribution |
| Barabasi-Albert | growth + preferential attachment | can a "rich get richer" rule reproduce the hubs? |
| Watts-Strogatz | ring lattice + random rewiring | can a lattice + shortcuts reproduce clustering *and* short paths? |

**The bug and the fix.** In the real graph `<k> = 2m/n` is about 3.4, so `m/n` is about 1.7 - a non-integer value. Both classic generators take integer parameters only:

- Barabasi-Albert adds `m_BA` edges per new node, so it can only produce about `m_BA * n` edges. `round(1.7) = 2` gives about 61k edges - 18% too many. The fix below uses a **fractional attachment parameter**: each new node attaches with 1 or 2 edges, with the choice made so the cumulative edge count tracks the target, while the attachment probability stays proportional to degree. The scale-free mechanism is unchanged; only the edge budget is enforced.
- `nx.watts_strogatz_graph` requires an even `k` (each node connects to `k/2` neighbors on each side). `<k> = 3.4` rounded down to the even value 2, which produced a bare ring with about 30.5k edges - 41% too few, and a ring has clustering of exactly 0 before rewiring, which defeats the whole point of the model. The fix builds a ring lattice, then adds chords at increasing distance to a random subset of nodes until the edge budget is met exactly, and only then rewires each edge with probability `p` (rewiring moves one endpoint of an edge, so the edge count never changes).
- The configuration model is kept as it was in the original script: draw a multigraph on the real degree sequence, then collapse parallel edges and drop self-loops. This *necessarily* loses a small number of edges and can't be size-matched without breaking the degree sequence - so we report the number actually achieved instead of hiding it.

Why this matters: both clustering and average path length are monotonic in edge density. A model with 18% more edges will look like it has shorter paths for reasons that have nothing whatsoever to do with its mechanism.

In [ ]:
def simple_configuration_graph(degree_sequence, seed):
    '''Configuration model, then collapse parallel edges and drop self-loops.

    Keeps the real degree sequence exactly up to the collapse; the resulting
    (slightly smaller) edge count is reported in the comparison table.
    '''
    multigraph = nx.configuration_model(degree_sequence, seed=seed)
    graph = nx.Graph(multigraph)
    graph.remove_edges_from(nx.selfloop_edges(graph))
    return graph


def _random_subset(repeated_nodes, k, rng):
    '''Pick k distinct nodes with probability proportional to degree.

    `repeated_nodes` contains each node once per incident edge, so a uniform
    draw from it is a degree-proportional draw - the standard preferential
    attachment trick used by networkx itself.
    '''
    targets = set()
    while len(targets) < k:
        targets.add(rng.choice(repeated_nodes))
    return targets


def size_matched_barabasi_albert(n, m_target, seed):
    '''Preferential attachment with a fractional attachment parameter.

    Textbook BA adds an integer number of edges per new node and therefore
    cannot hit a target of m/n = 1.7 edges per node. Here the number of edges
    added by each new node is chosen from the remaining edge budget, so the
    final graph has exactly m_target edges while attachment remains
    degree-proportional.
    '''
    rng = random.Random(seed)
    m_start = max(1, int(math.ceil(m_target / n)))
    graph = nx.empty_graph(m_start)
    repeated_nodes = list(range(m_start))
    for new_node in range(m_start, n):
        remaining_nodes = n - new_node
        needed = m_target - graph.number_of_edges()
        k = int(round(needed / remaining_nodes)) if remaining_nodes else 1
        k = max(1, min(k, new_node))
        targets = _random_subset(repeated_nodes, k, rng)
        graph.add_node(new_node)
        for target in targets:
            graph.add_edge(new_node, target)
        repeated_nodes.extend(targets)
        repeated_nodes.extend([new_node] * k)
    return graph


def size_matched_watts_strogatz(n, m_target, rewire_p, seed):
    '''Watts-Strogatz style small world with an exact edge budget.

    Step 1: ring lattice of nearest neighbours (n edges).
    Step 2: add chords at distance 2, 3, ... to a random subset of nodes until
            the edge budget is met exactly (this is what allows a non-even
            effective k, i.e. an average degree of 3.4 rather than 2 or 4).
    Step 3: rewire each edge with probability rewire_p by moving one endpoint
            to a random node - this creates the shortcuts that give the small
            world its short paths and never changes the edge count.
    '''
    rng = random.Random(seed)
    graph = nx.Graph()
    graph.add_nodes_from(range(n))
    for i in range(n):
        graph.add_edge(i, (i + 1) % n)

    distance = 2
    while graph.number_of_edges() < m_target and distance < n // 2:
        order = list(range(n))
        rng.shuffle(order)
        for i in order:
            if graph.number_of_edges() >= m_target:
                break
            j = (i + distance) % n
            if i != j and not graph.has_edge(i, j):
                graph.add_edge(i, j)
        distance += 1

    if graph.number_of_edges() > m_target:
        surplus = graph.number_of_edges() - m_target
        edges = list(graph.edges())
        rng.shuffle(edges)
        graph.remove_edges_from(edges[:surplus])

    for u, v in list(graph.edges()):
        if rng.random() >= rewire_p:
            continue
        for _ in range(32):
            w = rng.randrange(n)
            if w != u and not graph.has_edge(u, w):
                graph.remove_edge(u, v)
                graph.add_edge(u, w)
                break
    return graph


print('Model generators ready.')

## 8. Generating the models and verifying the size match

Now we build all four synthetic graphs on the same number of nodes as the real graph, and print the edge counts actually achieved side by side with the numbers the **old, incorrect** parameter choice would have produced. That makes the fix auditable rather than a claim in prose. Generation takes well under a minute; the configuration model is the slowest step.

In [ ]:
degree_sequence = [degree for _, degree in G.degree()]

models = {}
models['real_transit'] = G
models['erdos_renyi_gnm'] = nx.gnm_random_graph(N_REAL, M_REAL, seed=SEED)
models['configuration_degree_sequence'] = simple_configuration_graph(degree_sequence, SEED)
models['barabasi_albert_matched'] = size_matched_barabasi_albert(N_REAL, M_REAL, SEED)
models['watts_strogatz_matched'] = size_matched_watts_strogatz(N_REAL, M_REAL, WS_REWIRE_P, SEED)

print('Achieved sizes')
for name, graph in models.items():
    n_model = graph.number_of_nodes()
    m_model = graph.number_of_edges()
    print(f'  {name:32s} n={n_model:7,d}  m={m_model:7,d}  edges vs real = {m_model / M_REAL:6.1%}')

# What the previous (buggy) parameter choice would have produced, for contrast.
OLD_BA_M = max(1, int(round(M_REAL / N_REAL)))
OLD_BA_EDGES = OLD_BA_M * (N_REAL - OLD_BA_M)
old_ws_k = max(2, int(round(2 * M_REAL / N_REAL)))
OLD_WS_K = old_ws_k if old_ws_k % 2 == 0 else old_ws_k - 1
OLD_WS_EDGES = N_REAL * OLD_WS_K // 2
print()
print('Previous parameterisation (the bug this notebook fixes)')
print(f'  barabasi_albert m=round(m/n)={OLD_BA_M} would give about {OLD_BA_EDGES:,} edges '
      f'({OLD_BA_EDGES / M_REAL:.1%} of the real network)')
print(f'  watts_strogatz forced-even k={OLD_WS_K} would give exactly {OLD_WS_EDGES:,} edges '
      f'({OLD_WS_EDGES / M_REAL:.1%} of the real network)')

## 9. The comparison table

Every graph is now streamed through exactly the same measurement pipeline. The table reports, in order: the achieved size (`nodes`, `edges`, and the ratios against the real graph), a degree summary, connectivity, clustering (noting the method used), transitivity, and the sampling-based path statistics.

**Cost warning:** this is the expensive cell - `PATH_SOURCE_SAMPLES` BFS sweeps per graph plus exact clustering and transitivity, about 2-5 minutes in total. Reduce `PATH_SOURCE_SAMPLES` if you need a faster run.

Reading guide: `average_clustering` is the mean of the local clustering coefficient per node (it gives a lot of weight to low-degree nodes); `transitivity` is the global triangle ratio (it gives a lot of weight to hubs). Reporting both is deliberate - they disagree sharply in heavy-tailed graphs.

In [ ]:
comparison = pd.DataFrame([graph_property_row(name, graph) for name, graph in models.items()])
comparison.to_csv(TABLES / 'network_model_comparison.csv', index=False, encoding='utf-8-sig')
print('Written:', TABLES / 'network_model_comparison.csv')
comparison.round(4)

## 10. Degree distributions

The comparison table gives a single clustering number per graph; the degree distribution shows the whole shape. For each graph we tabulate P(k) and the complementary CDF P(K >= k), then summarize with:

- **mean / standard deviation / maximum degree** - how heterogeneous the graph is. In a Poisson graph the standard deviation is about the square root of the mean; in a scale-free graph the standard deviation is much larger and the maximum is far larger.
- **hill_alpha_kmin4** - a power-law exponent estimated by maximum likelihood over the tail `k >= 4`. This is a *descriptive-only* measure: `k_min` is fixed in advance rather than chosen optimally, and no goodness-of-fit test was run, so read it as "how heavy is the tail" rather than as evidence that the distribution is a power law.
- **ks_vs_real** - the Kolmogorov-Smirnov distance (the maximum gap between the empirical degree CDFs) between each model and the real graph. A smaller value means a better fit; the configuration model should be close to zero by construction, which doubles as a sanity check on the pipeline.

In [ ]:
def degree_distribution_frame(name, graph):
    '''Long-form degree distribution: P(k), CDF and CCDF for one graph.'''
    counts = Counter(degree for _, degree in graph.degree())
    n = graph.number_of_nodes()
    rows = []
    cumulative = 0
    for degree in sorted(counts):
        nodes = counts[degree]
        cumulative += nodes
        rows.append({'model': name, 'degree': degree, 'nodes': nodes,
                     'probability': nodes / n if n else 0.0,
                     'cumulative_probability': cumulative / n if n else 0.0,
                     'ccdf': 1.0 - (cumulative - nodes) / n if n else 0.0})
    return pd.DataFrame(rows)


def degree_cdf(degrees, max_k):
    counts = np.bincount(np.asarray(degrees, dtype=int), minlength=max_k + 1)
    return np.cumsum(counts) / counts.sum()


def ks_statistic(degrees_a, degrees_b):
    '''Kolmogorov-Smirnov distance between two empirical degree distributions.'''
    max_k = max(max(degrees_a), max(degrees_b))
    return float(np.max(np.abs(degree_cdf(degrees_a, max_k) - degree_cdf(degrees_b, max_k))))


def hill_alpha(degrees, k_min):
    '''MLE power-law exponent for the tail k >= k_min. Descriptive only.'''
    tail = [d for d in degrees if d >= k_min]
    if len(tail) < 20:
        return None
    total = sum(math.log(d / (k_min - 0.5)) for d in tail)
    if total <= 0:
        return None
    return 1.0 + len(tail) / total


distributions = pd.concat(
    [degree_distribution_frame(name, graph) for name, graph in models.items()],
    ignore_index=True)
distributions.to_csv(TABLES / 'degree_distributions.csv', index=False, encoding='utf-8-sig')

real_degrees = [degree for _, degree in G.degree()]
summary_rows = []
for name, graph in models.items():
    degrees = [degree for _, degree in graph.degree()]
    array = np.asarray(degrees)
    summary_rows.append({
        'model': name,
        'nodes': graph.number_of_nodes(),
        'edges': graph.number_of_edges(),
        'mean_degree': float(array.mean()),
        'std_degree': float(array.std()),
        'max_degree': int(array.max()),
        'share_degree_ge_10': float((array >= 10).mean()),
        'isolated_nodes': int((array == 0).sum()),
        'hill_alpha_kmin4': hill_alpha(degrees, K_MIN_TAIL),
        'ks_vs_real': 0.0 if name == 'real_transit' else ks_statistic(degrees, real_degrees),
    })

degree_summary = pd.DataFrame(summary_rows)
degree_summary.to_csv(TABLES / 'degree_distribution_summary.csv', index=False, encoding='utf-8-sig')
print('Written:', TABLES / 'degree_distributions.csv')
print('Written:', TABLES / 'degree_distribution_summary.csv')
degree_summary.round(4)

## 11. Figure - degree distributions on log-log axes

Two panels on log-log axes. The left panel shows the raw P(k), which is noisy in the tail because each high degree is represented by only a handful of stations. The right panel shows the CCDF, P(K >= k), which is the standard way to read a heavy tail: a straight line there indicates power-law-like behavior, while a sharp downward bend indicates an exponential cutoff. Comparing the real curve against Erdos-Renyi (Poisson, dropping off a cliff), Barabasi-Albert (the straightest line), and Watts-Strogatz (almost a delta function around the mean degree) is the fastest visual read of how heterogeneous the real network is.

In [ ]:
PALETTE = {
    'real_transit': '#111827',
    'erdos_renyi_gnm': '#2563eb',
    'configuration_degree_sequence': '#0f766e',
    'barabasi_albert_matched': '#dc2626',
    'watts_strogatz_matched': '#7c3aed',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in models:
    subset = distributions[(distributions['model'] == name) & (distributions['degree'] > 0)]
    colour = PALETTE.get(name, '#6b7280')
    axes[0].scatter(subset['degree'], subset['probability'], s=16, alpha=0.7,
                    color=colour, label=name)
    axes[1].plot(subset['degree'], subset['ccdf'].clip(lower=1e-6),
                 linewidth=1.8, color=colour, label=name)

axes[0].set_title('Degree distribution P(k), log-log')
axes[0].set_ylabel('P(k)')
axes[1].set_title('Complementary CDF P(K >= k), log-log')
axes[1].set_ylabel('P(K >= k)')
for ax in axes:
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Degree k')
    ax.grid(True, which='both', alpha=0.2)
axes[1].legend(fontsize=8, loc='lower left')
fig.suptitle('Real transit network vs. random network models: degree distribution')
fig.tight_layout()
fig.savefig(FIGURES / 'degree_distribution_comparison.png', dpi=180)
plt.show()
print('Written:', FIGURES / 'degree_distribution_comparison.png')

## 12. Figure - clustering and path length side by side

Three panels, one per central metric, because they live on completely different scales (clustering is in [0, 1], while path length is measured in tens of hops) and wouldn't be readable on shared axes. Since all five graphs now carry (almost) the same node and edge counts, the differences here can be attributed to structure rather than density.

In [ ]:
metric_specs = [
    ('average_clustering', 'Average clustering coefficient'),
    ('transitivity', 'Transitivity (global triangle ratio)'),
    ('approx_average_shortest_path', 'Approx. average shortest path (hops)'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
labels = comparison['model'].tolist()
colours = [PALETTE.get(name, '#6b7280') for name in labels]
for ax, (column, title) in zip(axes, metric_specs):
    values = pd.to_numeric(comparison[column], errors='coerce').fillna(0.0)
    ax.bar(range(len(labels)), values, color=colours)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.grid(True, axis='y', alpha=0.2)
    for x, value in enumerate(values):
        ax.annotate(f'{value:.3g}', (x, value), ha='center', va='bottom', fontsize=8)

fig.suptitle('Structural metrics at matched size (same nodes, same edge budget)')
fig.tight_layout()
fig.savefig(FIGURES / 'model_metric_comparison.png', dpi=180)
plt.show()
print('Written:', FIGURES / 'model_metric_comparison.png')

## 13. Figure - size-match audit

This figure exists purely to make the bug fix visible. It plots the edge count actually achieved by each model against the real graph (dashed line), together with two gray bars showing what the previous parameterization would have produced. Any bar that doesn't sit on the dashed line represents a remaining mismatch that the reader is entitled to know about - and in practice that's only the configuration model, which loses edges when parallel edges are collapsed.

In [ ]:
audit_labels = list(models.keys()) + ['barabasi_albert (old, m=2)', 'watts_strogatz (old, even k)']
audit_values = [graph.number_of_edges() for graph in models.values()] + [OLD_BA_EDGES, OLD_WS_EDGES]
audit_colours = [PALETTE.get(name, '#6b7280') for name in models] + ['#9ca3af', '#9ca3af']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(audit_labels)), audit_values, color=audit_colours)
ax.axhline(M_REAL, color='#111827', linestyle='--', linewidth=1.4,
           label=f'real edge count = {M_REAL:,}')
ax.set_xticks(range(len(audit_labels)))
ax.set_xticklabels(audit_labels, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Undirected edges')
ax.set_title('Size-match audit: achieved edge counts vs. the real network')
ax.grid(True, axis='y', alpha=0.2)
for x, value in enumerate(audit_values):
    ax.annotate(f'{value / M_REAL:.0%}', (x, value), ha='center', va='bottom', fontsize=8)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'size_match_check.png', dpi=180)
plt.show()
print('Written:', FIGURES / 'size_match_check.png')

## 14. Small-world diagnostics

The classic small-world test compares a graph to a random graph of the same size: in a small world, `C / C_random >> 1` (locally clustered) while `L / L_random` stays close to 1 (globally short). The small-world index is `sigma = (C / C_rand) / (L / L_rand)`, where `sigma > 1` is conventionally taken as evidence of a small world. Here the Erdos-Renyi graph plays the role of the random reference, which is exactly what it was made for. Caveat: `L` is a sampling-based estimate, and `C_random` is a very small number, so `sigma` is sensitive to noise in the denominator - treat it as an order of magnitude rather than an exact statistic.

In [ ]:
def safe_ratio(numerator, denominator):
    if numerator is None or denominator is None:
        return None
    try:
        if float(denominator) == 0.0:
            return None
        return float(numerator) / float(denominator)
    except (TypeError, ValueError):
        return None


indexed = comparison.set_index('model')
reference = indexed.loc['erdos_renyi_gnm']
c_random = reference['average_clustering']
l_random = reference['approx_average_shortest_path']

small_world_rows = []
for name in indexed.index:
    row = indexed.loc[name]
    c_ratio = safe_ratio(row['average_clustering'], c_random)
    l_ratio = safe_ratio(row['approx_average_shortest_path'], l_random)
    small_world_rows.append({
        'model': name,
        'clustering_C': row['average_clustering'],
        'avg_path_L': row['approx_average_shortest_path'],
        'C_over_C_random': c_ratio,
        'L_over_L_random': l_ratio,
        'small_world_sigma': safe_ratio(c_ratio, l_ratio),
    })

small_world = pd.DataFrame(small_world_rows)
small_world.to_csv(TABLES / 'small_world_ratios.csv', index=False, encoding='utf-8-sig')
print('Written:', TABLES / 'small_world_ratios.csv')
small_world.round(3)

## 15. Data-driven verdict

Rather than fixing conclusions in prose that could drift from reality on a rerun, this cell prints a verdict built from the numbers actually computed above: which model matched in size, which matched in degree distribution (the lowest KS distance), how far the real clustering is from the random baseline, and whether the real average path length is anywhere near that of the random graph.

In [ ]:
def fmt(value, digits=4):
    if value is None:
        return 'n/a'
    try:
        number = float(value)
    except (TypeError, ValueError):
        return str(value)
    if math.isnan(number):
        return 'n/a'
    return f'{number:.{digits}f}'


real_row = indexed.loc['real_transit']
ks_ranking = degree_summary[degree_summary['model'] != 'real_transit'].sort_values('ks_vs_real')
best_degree_model = ks_ranking.iloc[0]
sw_indexed = small_world.set_index('model')
real_sw = sw_indexed.loc['real_transit']

print('=== Size match ===')
for name in indexed.index:
    ratio = indexed.loc[name, 'edge_ratio_vs_real']
    node_ratio = indexed.loc[name, 'node_ratio_vs_real']
    print(f'  {name:32s} nodes {node_ratio:6.1%}   edges {ratio:6.1%}')

print()
print('=== Degree distribution ===')
for _, row in degree_summary.iterrows():
    print(f'  {row["model"]:32s} mean {row["mean_degree"]:5.2f}  std {row["std_degree"]:5.2f}  '
          f'max {int(row["max_degree"]):4d}  KS vs real {row["ks_vs_real"]:.4f}')
print(f'  Closest degree distribution to the real graph: {best_degree_model["model"]} '
      f'(KS = {best_degree_model["ks_vs_real"]:.4f})')

print()
print('=== Clustering and path length ===')
print(f'  real   C = {fmt(real_row["average_clustering"])}   L = {fmt(real_row["approx_average_shortest_path"], 2)}')
print(f'  random C = {fmt(c_random)}   L = {fmt(l_random, 2)}')
print(f'  real C / random C = {fmt(real_sw["C_over_C_random"], 1)}   '
      f'real L / random L = {fmt(real_sw["L_over_L_random"], 2)}   '
      f'sigma = {fmt(real_sw["small_world_sigma"], 1)}')

l_ratio_value = real_sw['L_over_L_random']
if l_ratio_value is not None and float(l_ratio_value) > 2:
    print('  -> The real network is far MORE clustered than random but its paths are also much '
          'longer than random: it is not a small world in the Watts-Strogatz sense.')
else:
    print('  -> The real network combines above-random clustering with near-random path lengths: '
          'small-world behaviour.')

print()
print('=== Stage outputs ===')
for path in sorted(TABLES.glob('*.csv')) + sorted(FIGURES.glob('*.png')):
    print(' ', path.relative_to(REPO))

## Conclusions

Read these against the numbers printed in the verdict cell - these are the structural conclusions the comparison supports.

1. **The size-match fix changes the conclusion, not just the appearance.** The old parameterization compared the real network against a Barabasi-Albert graph with about 18% too many edges and against a Watts-Strogatz graph with about 41% too few. Both clustering and path length change with density, so these gaps alone could produce differences on the order of the ones being interpreted. With the fractional-`m` BA generator and the chord-based Watts-Strogatz generator, every model except the configuration model now carries exactly the real node and edge count; the configuration model necessarily loses a small fraction of edges when parallel edges are collapsed, and that loss is reported in `edge_ratio_vs_real` rather than hidden.

2. **No single classic model reproduces the transit network.** Each captures one property and misses the rest: Erdos-Renyi matches on density but its clustering is essentially zero and its degrees are far more concentrated around the mean; the configuration model matches the degree distribution by construction (its KS distance is close to zero) and yet destroys the clustering, showing that the real triangles are *not* a byproduct of the degree sequence; Barabasi-Albert produces a heavy degree tail but too few triangles and unrealistically short paths; Watts-Strogatz produces clustering in abundance but a nearly uniform degree distribution that has no counterpart in any real transit system.

3. **The dominant constraint is geography, and none of the four models is aware of it.** A station graph is close to planar: stations connect to physical neighbors along roads, so degrees can't grow without bound and distances grow roughly with geographic distance. That's why the real average shortest path is far longer than in any random graph of the same size - the network is embedded in space, and is not a small world. Clustering much higher than the random baseline together with path lengths much higher than the random baseline is the signature of a spatially constrained, lattice-like graph.

4. **Stated limitations.** `L` and the diameter are sampled from `PATH_SOURCE_SAMPLES` BFS sources within the largest component, so they're estimates (the diameter in particular is a lower bound). `hill_alpha_kmin4` is a descriptive tail measure with a fixed cutoff and no goodness-of-fit test - it doesn't establish that the degree distribution is a power law, and given a maximum degree in the low tens, the real network is better described as moderately heavy-tailed than as scale-free. Finally, `sigma` divides by the very small clustering of the random graph and should be read as an order of magnitude only.